# Инференс: запрос → рекомендации

Пользователь (история прослушиваний + профиль) пишет запрос (на английском, как в датасете), в ответ — текст и 20 треков.
Вся логика в пакете `recsys/`, здесь только запуск.

* `experiments.ipynb` — почему такая выдача: разбор по шагам, сравнение вариантов, LLM
* `evaluate.py` — метрики на валидации
* `examples/candgen.ipynb` — проверка сервисов BM25 и HNSW руками

## 1. Настройки

По умолчанию — настоящий датасет и оба сервиса на сервере:

* `USE_CANDGEN = True` — BM25 и HNSW с сервера по HTTP (`configs/candgen.yaml`). Адреса и ключи — `BM25_URL`, `BM25_API_KEY`,
  `HNSW_URL`, `HNSW_API_KEY` (файл `.env` в корне или секреты проекта DataSphere). `False` — всё считается локально, без HNSW.
* `DATA = "crs"` — датасет из `DATA_DIR` (`docs/dataset.md`);
  `"synthetic"` — сгенерированные данные, работает без файлов (только чтобы проверить код).

In [4]:
USE_CANDGEN = True         # BM25 и HNSW с сервера; False — всё локально, без HNSW
DATA = "crs"               # настоящий датасет; "synthetic" — сгенерированные примеры, только чтобы проверить код
DATA_DIR = "music4all_crs"  # tracks_meta, test_public, test_public_queries, test_public_qrels
N_USERS = 300              # сколько пользователей загрузить (None — все, долго)

# адреса и ключи сервисов — из файла .env или секретов DataSphere; можно и здесь:
# import os; os.environ["BM25_URL"] = "http://<IP>:8000"; os.environ["BM25_API_KEY"] = "..."
# os.environ["HNSW_URL"] = "http://<IP>:8001"; os.environ["HNSW_API_KEY"] = "..."

In [ ]:
%pip install -q -r requirements.txt

## 2. Каталог и пайплайн

Загружается один раз. Для сервисов печатается, доступны ли они и какая версия индекса.

In [5]:
%load_ext autoreload
%autoreload 2

from dataclasses import replace

from recsys.config import load_config
from recsys.data import load_data
from recsys.pipeline import Pipeline
from recsys.schemas import Message

files = ["configs/default.yaml"] + (["configs/candgen.yaml"] if USE_CANDGEN else [])
cfg = load_config(files, {"data": {"source": DATA, "crs": {"dir": DATA_DIR, "n_users": N_USERS}}})
data = load_data(cfg)
pipe = Pipeline.from_config(cfg, data.catalog)

for source in pipe.retrievers:
    if getattr(source, "remote", False):
        print(source.name, source.health())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[data] crs: DataBundle(tracks=64016, requests: test_public=1354) за 1.3 c
bm25_genres {'url': 'http://84.201.172.104:8000', 'status': 'ok', 'index_version': 'ca4b49fbb8a5', 'n_items': 64016}
bm25_tags {'url': 'http://84.201.172.104:8000', 'status': 'ok', 'index_version': '09b208549fa3', 'n_items': 64016}
hnsw {'url': 'http://84.201.172.104:8001', 'status': 'ok', 'index_version': 'hnsw-v1/17', 'n_items': 64016}


## 3. Запрос

Пользователь из `test_public`: его история, профиль и настоящий запрос; известен трек, который он на самом деле
послушал (цель). `REQUEST` — номер запроса; `TEXT` — свой текст вместо запроса из датасета (история та же).

In [6]:
REQUEST = 0      # номер запроса в test_public (загружено N_USERS пользователей, до 5 запросов у каждого)
TEXT = None      # None — настоящий запрос из датасета (известен трек, который послушали); или свой текст:
# TEXT = "something calm for the evening, no rap"

user = data.requests[REQUEST]
# запрос нужного типа: user = next(r for r in data.requests if r.meta["query_type"] == "similar_to")
request = replace(user, dialog=[Message("user", TEXT)]) if TEXT else user
print("запрос:", request.dialog[-1].text)
if not TEXT:
    print("тип:", request.meta["query_type"], "| цель:", request.meta["artist"], "—", request.meta["title"],
          "| уже слушал:", not request.meta["is_new"])
print("треков в истории:", len(request.history))
print("профиль:", request.user_info[:200], "...")

запрос: fast high energy rock in minor key
тип: audio_attributes | цель: Therapy? — Trigger Inside | уже слушал: False
треков в истории: 392
профиль: Listening since 2009; 752 plays before this period, 392 distinct tracks, 264 artists. Favourite artists: Oh Wonder, The Prodigy, Ed Sheeran, The 1975, CHVRCHES, Take That, Röyksopp, Twenty One Pilots, ...


## 4. Ответ

In [7]:
response = pipe.run(request, debug=True)
print(response.text)
response.to_frame()

Hi! Here is what I picked (fast, high energy, rock, minor key):
1. Twenty One Pilots — Ride [rock, reggae, alternative hip hop] — matches: rock; you have listened to it before
2. Oh Wonder — Dazzle [electronic, pop, indie poptimism] — you have listened to it before
3. Oh Wonder — Without You [indie, indie pop, pop] — you have listened to it before
4. Oh Wonder — White Blood [indie, british, piano] — you have listened to it before
5. Oh Wonder — Drive [indie pop, indie, british] — you have listened to it before
6. Pitty — Medo [rock, pitty, rock nacional] — matches: rock
7. The 1975 — Somebody Else [pop, indie, synthpop] — you have listened to it before
8. Shaed — Trampoline [pop, electronic, minor key tonality] — matches: minor key; you have listened to it before
9. Evelyn Thomas — High Energy [disco, 80s, dance] — matches: high energy
10. Imagine Dragons — Believer [rock, alternative, imagine dragons] — matches: rock, minor key; you have listened to it before


,rank,track_id,artist,title,score,tags,reasons
0,1,0yECeqcY1EWblSB0,Twenty One Pilots,Ride,0.0339,"rock, reggae, alternative hip hop, indie, hip ...",matches: rock; you have listened to it before
1,2,xcRMLQyZmL7tnKoA,Oh Wonder,Dazzle,0.0219,"electronic, pop, indie poptimism, electronica,...",you have listened to it before
2,3,5uS8vWF3uTFohZRA,Oh Wonder,Without You,0.0202,"indie, indie pop, pop, 2015, british, love at ...",you have listened to it before
3,4,0bKz1eyk1PsbezU5,Oh Wonder,White Blood,0.0200,"indie, british, piano, pop, indie poptimism, soul",you have listened to it before
4,5,uIOzuNazoszupxf7,Oh Wonder,Drive,0.0188,"indie pop, indie, british, electropop, pop, 20...",you have listened to it before
5,6,1Gv6HyHiiQ0yEEf8,Pitty,Medo,0.0178,"rock, pitty, rock nacional, nx zero, chiaroscu...",matches: rock
6,7,vSCXYItx2du4zMg5,The 1975,Somebody Else,0.0170,"pop, indie, synthpop, atmospheric, the 1975, s...",you have listened to it before
7,8,oS5YPBH766qjYEYR,Shaed,Trampoline,0.0164,"pop, electronic, minor key tonality, vocal due...",matches: minor key; you have listened to it be...
8,9,FH6yMRmoBaMOZHEt,Evelyn Thomas,High Energy,0.0164,"disco, 80s, dance, hi nrg, party, female vocal...",matches: high energy
9,10,TEoUTsR0WANQyvaQ,Imagine Dragons,Believer,0.0161,"rock, alternative, imagine dragons, indie rock...","matches: rock, minor key; you have listened to..."


Если запрос из датасета: на каком месте трек, который пользователь послушал.

In [8]:
if not TEXT:
    target = request.target_ids[0]
    place = response.track_ids.index(target) + 1 if target in response.track_ids else "нет в top-20"
    print("цель:", request.meta["artist"], "—", request.meta["title"], "| место в выдаче:", place)

цель: Therapy? — Trigger Inside | место в выдаче: нет в top-20


## 5. Почему такая выдача

Как понят запрос (английские теги — именно они уходят в BM25) и сколько кандидатов дал каждый источник.
`relisten` — треки из своей истории, `bm25_*` и `hnsw` — сервисы на сервере, остальные считаются здесь.

In [9]:
print("понял как:", response.summary.text)
print("теги:", response.summary.include_tags, "| исключить:", response.summary.exclude_tags)
print("кандидатов:", response.debug["n_candidates"])

понял как: User wants fast, high energy, rock, minor key; energy: high; generally likes pop, electronic, indie pop, soul, rock, synthpop, house, indie rock.
теги: ['fast', 'high energy', 'rock', 'minor key'] | исключить: []
кандидатов: {'relisten': 100, 'title': 0, 'history': 200, 'audio': 200, 'popular': 100, 'bm25_genres': 200, 'bm25_tags': 200, 'hnsw': 200}


## 6. Что вернули сервисы

Первые кандидаты BM25 и HNSW с местом и оценкой. Цель отмечена `<<`. Если источник пуст — смотрите, какие слова ушли
в BM25 (раздел 5, «теги»), и проверьте их руками в `examples/candgen.ipynb`.

In [10]:
import pandas as pd

target = request.target_ids[0] if not TEXT else None
for name in ["bm25_genres", "bm25_tags", "hnsw"]:
    cands = response.debug["candidates"].get(name, [])
    print(f"{name}: {len(cands)} кандидатов")
    rows = [{"место": c.rank, "оценка": round(c.score, 3), "артист": data.catalog.row(c.track_id)["artist"],
             "трек": data.catalog.row(c.track_id)["title"], "": "<<" if c.track_id == target else ""}
            for c in cands[:10]]
    if rows:
        display(pd.DataFrame(rows))

bm25_genres: 200 кандидатов


,место,оценка,артист,трек,
0,1,2.196,The All-American Rejects,I For You,
1,2,2.196,Him,Buried Alive by Love,
2,3,2.196,Joni Mitchell,Dog Eat Dog,
3,4,2.196,Los Daniels,Quisiera Saber,
4,5,2.196,James Bay,When We Were on Fire,
5,6,2.196,Wilco,Sky Blue Sky,
6,7,2.196,Beck,Qué Onda Guero,
7,8,2.196,Pavement,The Hexx,
8,9,2.196,Liz Phair,Firewalker,
9,10,2.196,Mark Knopfler,Golden Heart,


bm25_tags: 200 кандидатов


,место,оценка,артист,трек,
0,1,11.370,Moderat,Seamonkey,
1,2,10.216,Janet Jackson,Dream Street,
2,3,8.447,Kelly Key,Pegue e Puxe,
3,4,8.447,Kelly Key,Por Causa De Você,
4,5,7.544,The Hellacopters,Hopeless Case of a Kid in Denial,
5,6,6.552,Prince,How Come U Don't Call Me Anymore,
6,7,5.891,Ween,I'm Dancing in the Show Tonight,
7,8,5.352,Twenty One Pilots,Ride,
8,9,4.903,Phish,Bouncing Around the Room,
9,10,4.444,A Rocket to the Moon,life of the party,


hnsw: 200 кандидатов


,место,оценка,артист,трек,
0,1,0.484,Evelyn Thomas,High Energy,
1,2,0.467,Linkin Park,High Voltage,
2,3,0.460,Smash Into Pieces,Higher,
3,4,0.459,Palm,Theme From Rock Island,
4,5,0.442,Welshly Arms,Love in a Minor Key,
5,6,0.439,Motörhead,Fast and Loose,
6,7,0.435,Metallica,Whiplash,
7,8,0.435,Manowar,Fast Taker,
8,9,0.432,Coldplay,High Speed,
9,10,0.425,Motörhead,Vibrator,


## 7. Следующая реплика

Диалог продолжается: уже показанное не повторяется, новые пожелания перекрывают старые.

In [11]:
request2 = request.next_turn(response, "now something more energetic")
response2 = pipe.run(request2)
print(response2.text)
response2.to_frame()

Hi! Here is what I picked (fast, high energy, rock, minor key, now, energetic):
1. Hozier — Take Me to Church [indie, soul, favorite track right now] — matches: rock, now; you have listened to it before
2. Orchestral Manoeuvres in the Dark — Genetic Engineering [80s, synthpop, electronic] — you listen to Orchestral Manoeuvres in the Dark
3. Take That — Happy Now [robbie williams, british, pop] — you have listened to it before
4. Queen — Don't Stop Me Now - Remastered 2011 [rock, queen, hard rock] — matches: rock
5. Matchbox Twenty — How Far We've Come [rock, alternative rock, alternative] — matches: rock, energetic; you have listened to it before
6. Iman Omari — Energy [electronic, electronica] — close to your taste: electronic
7. The Wombats — Moving To New York [indie, indie rock, british] — matches: rock, energetic; you have listened to it before
8. Austra — Energy [indie pop, synth pop, canadian] — matches: rock
9. The Black Eyed Peas — Pump It [dance, pop, black eyed peas] — match

,rank,track_id,artist,title,score,tags,reasons
0,1,DIenz1hGgjltkjQF,Hozier,Take Me to Church,0.0284,"indie, soul, favorite track right now, 2013, i...","matches: rock, now; you have listened to it be..."
1,2,K65sHXYw6Vm2H2Uk,Orchestral Manoeuvres in the Dark,Genetic Engineering,0.0246,"80s, synthpop, electronic, pop, synth pop, new...",you listen to Orchestral Manoeuvres in the Dark
2,3,V7UlqE12DCKSsVrd,Take That,Happy Now,0.0239,"robbie williams, british, pop, britpop, boyban...",you have listened to it before
3,4,gYxWozyzUYjYWISO,Queen,Don't Stop Me Now - Remastered 2011,0.0184,"rock, queen, hard rock, fode, british, classic...",matches: rock
4,5,3PtdXqM8u7cL8pe5,Matchbox Twenty,How Far We've Come,0.0164,"rock, alternative rock, alternative, matchbox ...","matches: rock, energetic; you have listened to..."
5,6,VZKe1A2lpbmSAS3G,Iman Omari,Energy,0.0164,"electronic, electronica",close to your taste: electronic
6,7,8Qn0ebFgBw3o3zzA,The Wombats,Moving To New York,0.0161,"indie, indie rock, british, britpop, alternati...","matches: rock, energetic; you have listened to..."
7,8,LFwG21lROsb7K1gm,Austra,Energy,0.0161,"indie pop, synth pop, canadian, female vocalis...",matches: rock
8,9,vK6gwLuw49kscUFS,The Black Eyed Peas,Pump It,0.0159,"dance, pop, black eyed peas, hip hop, rap, rnb",matches: energetic; you have listened to it be...
9,10,QBhbI5wUUzoV9MS1,Pouya,Energy,0.0159,"rap, hip hop",
